In [2]:
!pip install -q mistralai pandas duckdb
import os, time, requests
import json, re
#from getpass import getpass
import pandas as pd
import duckdb
from mistralai.client import Mistral
from google.colab import userdata
os.environ["MISTRAL_API_KEY"] = userdata.get("climate_api_key")

# Initialises AI model

In [3]:

key = os.environ["MISTRAL_API_KEY"]
for m in ["mistral-small-2603", "ministral-8b-latest", "open-mistral-nemo"]:
    r = requests.post("https://api.mistral.ai/v1/chat/completions",
        headers={"Authorization": f"Bearer {key}"},
        json={"model": m, "messages": [{"role": "user", "content": "Hi"}]})
    print(m, r.status_code, r.text[:120])
    print({k: v for k, v in r.headers.items() if "limit" in k.lower() or "retry" in k.lower()})
    print()

mistral-small-2603 429 {"object":"error","message":"Rate limit exceeded","type":"rate_limited","param":null,"code":"1300","raw_status_code":429
{'x-ratelimit-limit-req-minute': '0', 'x-ratelimit-remaining-req-minute': '0'}

ministral-8b-latest 200 {"id":"980271181d3c4222aa2b9ae4549515c7","created":1795472848,"model":"ministral-8b-latest","usage":{"prompt_tokens":4,"
{'x-max-retry-attempts-reached': 'false', 'x-ratelimit-limit-tokens-minute': '40000', 'x-ratelimit-remaining-tokens-minute': '39953', 'x-ratelimit-tokens-query-cost': '47', 'x-ratelimit-limit-req-minute': '2', 'x-ratelimit-remaining-req-minute': '1'}

open-mistral-nemo 200 {"id":"282d88391ed749e38c3825b1f2be79af","created":1795472848,"model":"ministral-8b-2512","usage":{"prompt_tokens":4,"to
{'x-max-retry-attempts-reached': 'false', 'x-ratelimit-limit-tokens-minute': '40000', 'x-ratelimit-remaining-tokens-minute': '39904', 'x-ratelimit-tokens-query-cost': '49', 'x-ratelimit-limit-req-minute': '2', 'x-ratelimit-remaining-req

In [4]:

client = Mistral(api_key = os.environ["MISTRAL_API_KEY"])
def ask(prompt, model="ministral-8b-latest", tries=4):
    for i in range(tries):
        try:
            r = client.chat.complete(model=model,
                messages=[{"role": "user", "content": prompt}])
            return r.choices[0].message.content
        except Exception as e:
            if "429" in str(e):
                time.sleep(2 ** i * 5)  # 5s, 10s, 20s, 40s
            else:
                raise
    raise RuntimeError("Still rate limited")

print(ask("In one sentence, what is a temperature anomaly?"))

A **temperature anomaly** is the deviation of a given location’s average temperature from the long-term historical average (typically over 30 years), measured in degrees (e.g., above or below normal).


# Load data

In [5]:

base = "https://datahub.io/core/global-temp/_r/-/data/"
annual = pd.read_csv(base + "annual.csv")
monthly = pd.read_csv(base + "monthly.csv")
print(annual.shape, monthly.shape)
print(annual.columns.tolist())
print(monthly.columns.tolist())


(323, 3) (3871, 3)
['Source', 'Year', 'Mean']
['Source', 'Year', 'Mean']


In [6]:
print(annual.groupby("Source")["Year"].agg(["min","max","count"]))
print(monthly.head(3))
print(monthly["Source"].value_counts())
print(annual.sort_values("Year").tail(4))

          min   max  count
Source                    
GCAG     1850  2026    177
GISTEMP  1880  2025    146
  Source     Year    Mean
0   GCAG  1850-01 -0.7337
1   GCAG  1850-02 -0.3604
2   GCAG  1850-03 -0.6271
Source
GCAG       2119
GISTEMP    1752
Name: count, dtype: int64
      Source  Year    Mean
319     GCAG  2024  1.1690
320  GISTEMP  2025  1.1850
321     GCAG  2025  1.0480
322     GCAG  2026  1.0421


In [7]:
monthly["Year"] = monthly["Year"].astype(str)
print(monthly.groupby("Source")["Year"].agg(["min", "max", "count"]))
print("Months in 2026:", monthly[monthly["Year"].str.startswith("2026")]
      .groupby("Source").size().to_dict())
print("Missing:", annual.isna().sum().sum(), monthly.isna().sum().sum())

             min      max  count
Source                          
GCAG     1850-01  2026-07   2119
GISTEMP  1880-01  2025-12   1752
Months in 2026: {'GCAG': 7}
Missing: 0 0


In [8]:
con = duckdb.connect()

con.execute("""
CREATE OR REPLACE TABLE monthly_anomaly AS
SELECT
  Source AS source,
  CAST(strptime(Year, '%Y-%m') AS DATE) AS month_date,
  CAST(substr(Year, 1, 4) AS INTEGER) AS year,
  CAST(substr(Year, 6, 2) AS INTEGER) AS month,
  CAST(substr(Year, 1, 4) AS INTEGER) // 10 * 10 AS decade,
  Mean AS anomaly_c
FROM monthly
""")

con.execute("""
CREATE OR REPLACE TABLE annual_anomaly AS
SELECT
  a.Source AS source,
  CAST(a.Year AS INTEGER) AS year,
  CAST(a.Year AS INTEGER) // 10 * 10 AS decade,
  a.Mean AS anomaly_c,
  COALESCE(m.n, 0) AS months_of_data,
  COALESCE(m.n, 0) = 12 AS is_complete_year
FROM annual a
LEFT JOIN (
  SELECT source, year, COUNT(*) AS n
  FROM monthly_anomaly GROUP BY source, year
) m ON m.source = a.Source AND m.year = CAST(a.Year AS INTEGER)
""")

print(con.sql("DESCRIBE annual_anomaly"))
print(con.sql("DESCRIBE monthly_anomaly"))
print(con.sql("""
  SELECT * FROM annual_anomaly
  WHERE NOT is_complete_year ORDER BY source, year
"""))
print(con.sql("""
  SELECT source, COUNT(*) AS n FROM annual_anomaly GROUP BY source
"""))

┌──────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│   column_name    │ column_type │  null   │   key   │ default │  extra  │
│     varchar      │   varchar   │ varchar │ varchar │ varchar │ varchar │
├──────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ source           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ year             │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ decade           │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ anomaly_c        │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ months_of_data   │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ is_complete_year │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
└──────────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

┌─────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│ column_name │ column_type │  null   │   key   │ default │  extra  │
│   varchar   │   varchar   │ varc

In [9]:
SCHEMA = """
You can query a DuckDB database with two tables of GLOBAL temperature
ANOMALIES. Anomalies are differences in degrees Celsius from a baseline
average. They are NOT absolute temperatures.

TABLE annual_anomaly  -- one row per source per year
  source           VARCHAR  'GISTEMP' or 'GCAG' (exact, uppercase)
  year             INTEGER  e.g. 2025
  decade           INTEGER  start year of decade, e.g. 1990 = 1990-1999
  anomaly_c        DOUBLE   annual mean anomaly in degrees C
  months_of_data   BIGINT   number of months behind the annual value
  is_complete_year BOOLEAN  false if the year has fewer than 12 months

TABLE monthly_anomaly  -- one row per source per month
  source      VARCHAR  'GISTEMP' or 'GCAG'
  month_date  DATE     first day of the month, e.g. 2025-01-01
  year        INTEGER
  month       INTEGER  1-12
  decade      INTEGER
  anomaly_c   DOUBLE   monthly mean anomaly in degrees C

SOURCES (two independent datasets, stored in the same tables):
  GISTEMP: NASA GISS. Baseline 1951-1980. Covers 1880-01 to 2025-12.
  GCAG:    NOAA Climate at a Glance. Baseline 20th-century average.
           Covers 1850-01 to 2026-07.

IMPORTANT DATA FACTS:
  - The two sources use different baselines, so their values for the
    same year differ (2025: GISTEMP 1.19, GCAG 1.05). Never average,
    sum or directly compare values across sources. Always filter on
    one source, or show both side by side.
  - Before 1880, only GCAG has data.
  - GCAG 2026 is a partial year (January-July only). Its annual value
    is not comparable with full years. Use is_complete_year = true for
    year-on-year comparisons.
  - The 2020s decade is incomplete (2020 onwards only). Say so when
    comparing decades.
  - The data contains no absolute temperatures, no regional or country
    data, and no forecasts.
"""
print(len(SCHEMA.split()), "words")

259 words


# AI functions/tools

In [10]:
r = client.chat.complete(
    model="ministral-8b-2512",
    messages=[
        {"role": "system", "content": SCHEMA},
        {"role": "user", "content":
         "Return only a DuckDB SQL query, no explanation, no code "
         "fences: which decade had the highest average anomaly "
         "according to GISTEMP?"},
    ],
)
sql = r.choices[0].message.content.strip().strip("`")
sql = sql.removeprefix("sql").strip()
print(sql)
print(con.sql(sql))

SELECT decade, AVG(anomaly_c) AS avg_anomaly
FROM annual_anomaly
WHERE source = 'GISTEMP' AND is_complete_year = true
GROUP BY decade
ORDER BY avg_anomaly DESC
LIMIT 1;
┌────────┬────────────────────┐
│ decade │    avg_anomaly     │
│ int32  │       double       │
├────────┼────────────────────┤
│   2020 │ 1.0627833333333332 │
└────────┴────────────────────┘



In [11]:
def run_sql(query: str) -> str:
    q = query.strip().rstrip(";")
    ok = re.match(r"(?is)^\s*(select|with)\b", q)
    if not ok or ";" in q:
        return json.dumps({"error": "Only one read-only SELECT query is allowed."})
    try:
        df = con.sql(q).df()
        return df.head(50).to_json(orient="records", date_format="iso")
    except Exception as e:
        return json.dumps({"error": str(e)})

In [12]:
tools = [{
  "type": "function",
  "function": {
    "name": "run_sql",
    "description": "Run one read-only DuckDB SELECT query on the climate tables. Returns rows as JSON, or an error.",
    "parameters": {
      "type": "object",
      "properties": {
        "query": {"type": "string",
                  "description": "A single DuckDB SELECT query"}},
      "required": ["query"]}}}]

In [13]:
print(run_sql("SELECT source, COUNT(*) AS n FROM annual_anomaly GROUP BY source"))
print(run_sql("DROP TABLE annual_anomaly"))
print(run_sql("SELECT nope FROM annual_anomaly"))

[{"source":"GCAG","n":177},{"source":"GISTEMP","n":146}]
{"error": "Only one read-only SELECT query is allowed."}
{"error": "Binder Error: Referenced column \"nope\" not found in FROM clause!\nCandidate bindings: \"is_complete_year\", \"source\", \"anomaly_c\""}


In [14]:
RULES = """
RULES:
- Every number in your answer must appear in a run_sql result from this
  conversation. If you did not query it, do not state it.
- If the user names no source, query both and show each value side by
  side with its baseline. Never average, sum or subtract across sources.
- If asked to combine sources, explain why not, then show each source's
  value side by side.
- If the data cannot answer (absolute temperatures, regions, countries),
  say so plainly and explain what the data contains. You may offer
  related anomaly values, but only after querying them.
- The data has no forecasts. For future years, say so; never predict.
- Partial years are not comparable with full years. Do not say which is
  warmer; explain why.
- If run_sql returns an error, fix the query and try again.
- Final answer: 1-3 plain-English sentences, 2 decimals, degrees C.
  Name the source and its baseline.
- Mention a caveat only if it applies to the rows you returned.
- Do all calculations (averages, differences, counts, rankings) in SQL.
  Never calculate yourself.
- Only call a year partial if a returned row has is_complete_year = false
"""
SYSTEM = SCHEMA + RULES

In [15]:
NUM = re.compile(r"\d+\.\d+")

def collect(obj, out):
    if isinstance(obj, bool):
        return
    if isinstance(obj, (int, float)):
        out.append(abs(obj))
    elif isinstance(obj, dict):
        for v in obj.values(): collect(v, out)
    elif isinstance(obj, list):
        for v in obj: collect(v, out)

FEEDBACK = "These numbers in your answer"

def unsupported(answer, messages):
    allowed = [float(x) for x in NUM.findall(SYSTEM)]
    for m in messages:
        if not isinstance(m, dict):
            continue  # model drafts in this call: never trusted
        c = m.get("content") or ""
        if m["role"] == "tool":
            try: collect(json.loads(c), allowed)
            except Exception: pass
        elif (m["role"] == "user" and not c.startswith(FEEDBACK)) or \
             (m["role"] == "assistant" and not c.startswith("⚠")):
            allowed += [float(x) for x in NUM.findall(c)]
    bad = []
    for s in NUM.findall(answer or ""):
        x = float(s)
        tol = 0.5 * 10 ** -len(s.split(".")[1]) + 1e-9
        if not any(abs(x - a) <= tol for a in allowed):
            bad.append(s)
    return bad

In [16]:
def call(messages, model="ministral-8b-2512", tries=4):
    for i in range(tries):
        try:
            return client.chat.complete(model=model,
                messages=messages, tools=tools, tool_choice="auto")
        except Exception as e:
            if "429" in str(e):
                time.sleep(2 ** i * 15)
            else:
                raise
    raise RuntimeError("Still rate limited")

In [17]:
def chat(question, max_steps=6, history=None):
    messages = [{"role": "system", "content": SYSTEM}, *(history or []),
                {"role": "user", "content": question}]
    retried = False
    for _ in range(max_steps):
        msg = call(messages).choices[0].message
        messages.append(msg)
        if not msg.tool_calls:
            bad = unsupported(msg.content, messages)
            if not bad:
                return msg.content, messages
            print("CHECK FAILED:", bad)
            if retried:
                return f"⚠ Unverified numbers {bad}: " + msg.content, messages
            retried = True
            messages.append({"role": "user", "content":
                f"These numbers in your answer are not in any run_sql result: {bad}. "
                "Query them with run_sql or remove them, then answer again."})
            continue
        for tc in msg.tool_calls:
            args = tc.function.arguments
            if isinstance(args, str):
                args = json.loads(args)
            print("SQL:", args["query"])
            messages.append({"role": "tool", "name": tc.function.name,
                "content": run_sql(args["query"]), "tool_call_id": tc.id})
    return "Stopped: too many steps.", messages

# Questions

In [25]:
answer, msgs = chat("Which decade had the highest average anomaly according to GISTEMP?")
print(answer)

SQL: SELECT 
  decade,
  AVG(anomaly_c) AS avg_anomaly
FROM 
  annual_anomaly
WHERE 
  source = 'GISTEMP'
  AND is_complete_year = true
GROUP BY 
  decade
ORDER BY 
  avg_anomaly DESC
LIMIT 1;
The 2020s decade had the highest average anomaly of **1.06°C** according to **GISTEMP (baseline 1951–1980)**. However, this decade is incomplete (only 2020–2025 data are available), so the value may increase slightly with additional years.


In [28]:
answer, msgs = chat("What was the average temperature in 1990?")
print(answer)

I cannot provide the average temperature in 1990 because the data only contains **temperature anomalies** (differences from a baseline average), not absolute temperatures. The tables show how much warmer or cooler each year was compared to a reference period, but they do not give actual temperature values.

Would you like the **temperature anomaly** for 1990 from a specific source (e.g., GISTEMP or GCAG)? Let me know!


In [27]:
answer, msgs = chat("What was the warmest year?")
print(answer)

SQL: SELECT year, anomaly_c FROM annual_anomaly WHERE source = 'GISTEMP' ORDER BY anomaly_c DESC LIMIT 1;
The warmest year in the **GISTEMP** dataset (baseline 1951–1980) was **2024**, with an annual temperature anomaly of **1.28°C**. This is the highest value available in the dataset.


In [18]:
G = "source='GISTEMP'"
tests = [
 dict(id=1, cat="lookup", q="What was the GISTEMP anomaly in 1998?",
  sql=f"SELECT anomaly_c FROM annual_anomaly WHERE {G} AND year=1998",
  pass_if="value ±0.01"),
 dict(id=2, cat="lookup", q="What was the GCAG anomaly in 1850?",
  sql="SELECT anomaly_c FROM annual_anomaly WHERE source='GCAG' AND year=1850",
  pass_if="value ±0.01"),
 dict(id=3, cat="monthly", q="Which month in 2023 had the highest anomaly according to GISTEMP?",
  sql=f"SELECT month, anomaly_c FROM monthly_anomaly WHERE {G} AND year=2023 ORDER BY anomaly_c DESC LIMIT 1",
  pass_if="right month and value"),
 dict(id=4, cat="aggregate", q="What was the average GISTEMP anomaly in the 1990s?",
  sql=f"SELECT AVG(anomaly_c) FROM annual_anomaly WHERE {G} AND decade=1990",
  pass_if="value ±0.01"),
 dict(id=5, cat="ranking", q="What were the 3 warmest years according to NOAA's GCAG?",
  sql="SELECT year, anomaly_c FROM annual_anomaly WHERE source='GCAG' AND is_complete_year ORDER BY anomaly_c DESC LIMIT 3",
  pass_if="right 3 years; partial 2026 excluded or flagged"),
 dict(id=6, cat="compare", q="How much warmer was 2020 than 1980 according to GISTEMP?",
  sql=f"SELECT MAX(CASE WHEN year=2020 THEN anomaly_c END) - MAX(CASE WHEN year=1980 THEN anomaly_c END) AS diff FROM annual_anomaly WHERE {G}",
  pass_if="difference ±0.01"),
 dict(id=7, cat="window", q="Which decade had the biggest increase in average anomaly compared with the previous decade, according to GISTEMP?",
  sql=f"WITH d AS (SELECT decade, AVG(anomaly_c) a FROM annual_anomaly WHERE {G} AND is_complete_year GROUP BY decade) SELECT decade, a - LAG(a) OVER (ORDER BY decade) AS jump FROM d ORDER BY jump DESC NULLS LAST LIMIT 1",
  pass_if="right decade; flags 2020s incomplete if it's the answer; no speculation"),
 dict(id=8, cat="count", q="How many years had an anomaly above 1.0 °C according to GISTEMP?",
  sql=f"SELECT COUNT(*) FROM annual_anomaly WHERE {G} AND anomaly_c > 1.0",
  pass_if="exact count"),
 dict(id=9, cat="threshold", q="What was the first year GISTEMP recorded an anomaly above 0.5 °C?",
  sql=f"SELECT MIN(year) FROM annual_anomaly WHERE {G} AND anomaly_c > 0.5",
  pass_if="exact year"),
]

In [19]:
tests += [
 dict(id=10, cat="trap_partial", q="Was 2026 warmer than 2025?",
  sql="SELECT source, year, anomaly_c, is_complete_year FROM annual_anomaly WHERE year IN (2025, 2026) ORDER BY source, year",
  pass_if="says 2026 is partial (GCAG, Jan-Jul) and not comparable"),
 dict(id=11, cat="trap_mixing", q="What is the average anomaly for 2025 across both datasets?",
  sql="SELECT source, anomaly_c FROM annual_anomaly WHERE year=2025",
  pass_if="refuses to average; shows both side by side"),
 dict(id=12, cat="trap_absolute", q="How hot was the Earth in 2010 in degrees Celsius?",
  sql=None, pass_if="explains anomalies; no invented absolute temperature"),
 dict(id=13, cat="unanswerable", q="What was the temperature anomaly in Sweden in 2018?",
  sql=None, pass_if="says no regional data; no invented number"),
 dict(id=14, cat="unanswerable", q="What will the anomaly be in 2030?",
  sql=None, pass_if="says no forecasts; no prediction"),
 dict(id=15, cat="no_data", q="What was the GISTEMP anomaly in 1870?",
  sql="SELECT source, MIN(year) AS first_year FROM annual_anomaly GROUP BY source",
  pass_if="says GISTEMP starts in 1880; may offer GCAG"),
 dict(id=16, cat="ambiguous", q="What was the anomaly in 2015?",
  sql=f"SELECT source, anomaly_c FROM annual_anomaly WHERE year=2015 ORDER BY source",
  pass_if="no source named: shows GISTEMP and GCAG side by side, each with its baseline (GISTEMP 1951–80, GCAG 20th-century avg); doesn't average or pick one; values ±0.01"),
]

In [32]:
for t in tests:
    t["expected"] = (con.sql(t["sql"]).df().round(4).to_dict("records")
                     if t["sql"] else None)
pd.DataFrame(tests).to_json("test_set_v2.json", orient="records", indent=2)
print(pd.DataFrame(tests)[["id", "cat", "expected"]].to_string())

    id            cat                                                                                                                                                                                                                                               expected
0    1         lookup                                                                                                                                                                                                                                [{'anomaly_c': 0.6083}]
1    2         lookup                                                                                                                                                                                                                               [{'anomaly_c': -0.4265}]
2    3        monthly                                                                                                                                                                            

In [20]:
tests = pd.read_json("test_set_v2.json").to_dict("records")

In [34]:
def get_sqls(msgs):
    out = []
    for m in msgs:
        for tc in (getattr(m, "tool_calls", None) or []):
            a = tc.function.arguments
            a = json.loads(a) if isinstance(a, str) else a
            out.append(a["query"])
    return out

results = []
for t in tests:
    try:
        ans, msgs = chat(t["q"])
    except Exception as e:
        ans, msgs = f"ERROR: {e}", []
    results.append(dict(id=t["id"], cat=t["cat"], q=t["q"],
        answer=ans, sql=get_sqls(msgs),
        expected=t["expected"], pass_if=t["pass_if"]))
    pd.DataFrame(results).to_json("run_v4.json",
        orient="records", indent=2)
    print(t["id"], "done")
    time.sleep(30)

SQL: SELECT anomaly_c FROM annual_anomaly WHERE source = 'GISTEMP' AND year = 1998;
1 done
SQL: SELECT anomaly_c FROM annual_anomaly WHERE source = 'GCAG' AND year = 1850;
2 done
SQL: SELECT month_date, anomaly_c FROM monthly_anomaly WHERE source = 'GISTEMP' AND year = 2023 ORDER BY anomaly_c DESC LIMIT 1;
3 done
SQL: SELECT AVG(anomaly_c) AS avg_anomaly_1990s
FROM annual_anomaly
WHERE source = 'GISTEMP' AND decade = 1990;
4 done
SQL: SELECT 
  year, 
  anomaly_c 
FROM 
  monthly_anomaly 
WHERE 
  source = 'GCAG' 
  AND is_complete_year = true
ORDER BY 
  anomaly_c DESC
LIMIT 3;
SQL: SELECT 
  year, 
  anomaly_c 
FROM 
  annual_anomaly 
WHERE 
  source = 'GCAG' 
  AND is_complete_year = true
ORDER BY 
  anomaly_c DESC
LIMIT 3;
5 done
SQL: SELECT 
    year,
    anomaly_c AS anomaly_2020,
    (SELECT anomaly_c FROM annual_anomaly WHERE source = 'GISTEMP' AND year = 1980) AS anomaly_1980
FROM 
    annual_anomaly
WHERE 
    source = 'GISTEMP' AND year = 2020;
CHECK FAILED: ['0.75']
SQL: SE

# Gradio

In [21]:
!pip install -q gradio
import gradio as gr, inspect
print("gradio", gr.__version__)
print(inspect.getsource(chat))

gradio 6.31.0
def chat(question, max_steps=6, history=None):
    messages = [{"role": "system", "content": SYSTEM}, *(history or []),
                {"role": "user", "content": question}]
    retried = False
    for _ in range(max_steps):
        msg = call(messages).choices[0].message
        messages.append(msg)
        if not msg.tool_calls:
            bad = unsupported(msg.content, messages)
            if not bad:
                return msg.content, messages
            print("CHECK FAILED:", bad)
            if retried:
                return f"⚠ Unverified numbers {bad}: " + msg.content, messages
            retried = True
            messages.append({"role": "user", "content":
                f"These numbers in your answer are not in any run_sql result: {bad}. "
                "Query them with run_sql or remove them, then answer again."})
            continue
        for tc in msg.tool_calls:
            args = tc.function.arguments
            if isinstance(args, str):
    

In [26]:
def sql_used(messages):
    qs = []
    for m in messages:
        for tc in (getattr(m, "tool_calls", None) or []):
            a = tc.function.arguments
            a = json.loads(a) if isinstance(a, str) else a
            qs.append(a["query"].strip())
    return qs

MAX_TURNS = 3

def to_text(c):
    if isinstance(c, list):
        return " ".join(p.get("text", "") if isinstance(p, dict) else str(p) for p in c)
    return str(c or "")

def clean_history(history):
    out = [{"role": m["role"],
            "content": to_text(m["content"]).split("\n\n**SQL run:**")[0]}
           for m in history if m["role"] in ("user", "assistant")]
    out = out[-2 * MAX_TURNS:]
    while out and out[0]["role"] != "user":
        out.pop(0)
    return out

def respond(message, history):
    hist = clean_history(history)
    print("history sent:", [(m["role"], m["content"][:40]) for m in hist])
    answer, messages = chat(message, history=hist)
    qs = sql_used(messages)
    if qs:
        answer += "\n\n**SQL run:**\n" + "\n".join(f"```sql\n{q}\n```" for q in qs)
    return answer

if "demo" in globals(): demo.close()
demo = gr.ChatInterface(fn=respond, title="Climate Data Assistant",
    description="Ask about global temperature anomalies (NASA GISTEMP, "
                "NOAA GCAG). Answers can take 1–2 min on the free tier.",
    examples=["How hot was the Earth in 2010?",
              "What was the anomaly in March?"])
demo.launch(show_error=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ed940a3d20b272ab6e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [23]:
demo.close()

Closing server running on port: 7860


In [27]:
print(run_sql("SELECT source, anomaly_c FROM monthly_anomaly "
              "WHERE year = 2010 AND month = 3 ORDER BY source"))

[{"source":"GCAG","anomaly_c":0.8514},{"source":"GISTEMP","anomaly_c":0.92}]


# Error Checking

In [29]:
_orig_run_sql = run_sql
state = {"broken": False}

def run_sql(query):
    if not state["broken"]:
        state["broken"] = True
        query = query.replace("anomaly_c", "anomaly")
    return _orig_run_sql(query)

answer, msgs = chat("What was the warmest year according to GISTEMP?")
print(answer)

SQL: SELECT year, anomaly_c FROM annual_anomaly WHERE source = 'GISTEMP' ORDER BY anomaly_c DESC LIMIT 1;
SQL: SELECT year, anomaly_c FROM annual_anomaly WHERE source = 'GISTEMP' ORDER BY anomaly_c DESC LIMIT 1;
According to **GISTEMP** (baseline 1951–1980), the warmest year was **2024**, with an annual mean anomaly of **1.28°C**. This is the most recent full-year value available.


In [30]:
for m in msgs:
    if isinstance(m, dict) and m["role"] == "tool":
        print(m["content"][:150])

{"error": "Binder Error: Referenced column \"anomaly\" not found in FROM clause!\nCandidate bindings: \"anomaly_c\", \"is_complete_year\""}
[{"year":2024,"anomaly_c":1.2817}]


In [31]:
run_sql = _orig_run_sql

In [25]:
for q_ in ["How many years had an anomaly above 1.0 °C according to GISTEMP?",
           "How hot was the Earth in 2010 in degrees Celsius?"]:
    print(chat(q_)[0], "\n")
    time.sleep(30)

SQL: SELECT COUNT(*) AS years_above_1_c FROM annual_anomaly WHERE source = 'GISTEMP' AND anomaly_c > 1.0 AND is_complete_year = true;
CHECK FAILED: ['1.22', '1.03']
SQL: SELECT year, anomaly_c FROM annual_anomaly WHERE source = 'GISTEMP' AND anomaly_c > 1.0 AND is_complete_year = true ORDER BY year DESC;
According to GISTEMP (baseline 1951–1980), **5 years** had an annual temperature anomaly above 1.00 °C. The most recent was 2024 (1.28 °C), and the earliest was 2016 (1.01 °C). 

CHECK FAILED: ['0.62', '0.65']
SQL: SELECT source, anomaly_c FROM annual_anomaly WHERE year = 2010;
The Earth's temperature anomaly in 2010 was:

- **GISTEMP (baseline 1951–1980):** **+0.72 °C**
- **GCAG (baseline 20th-century average):** **+0.67 °C**

*(Note: These are anomalies, not absolute temperatures.)* 



In [22]:
q = [{"role": "system", "content": SYSTEM},
     {"role": "user", "content": "How many years above 1.0?"}]
print(unsupported("5 years above 1.00 °C; most recent 2023 at 1.26 °C.",
      q + [{"role": "tool", "content": '[{"n": 5}]'}]))
print(unsupported("1998 was 0.61 °C.",
      q + [{"role": "tool", "content": '[{"anomaly_c": 0.6083}]'}]))

['1.26']
[]


In [29]:
print([t["pass_if"] for t in tests if t["id"] == 16])

['uses GISTEMP and says so; value ±0.01']


In [30]:
for t in tests:
    if int(t["id"]) == 16:
        t["sql"] = "SELECT source, anomaly_c FROM annual_anomaly WHERE year=2015 ORDER BY source"
        t["pass_if"] = "shows both sources side by side with baselines; values ±0.01; no average"
        t["expected"] = con.sql(t["sql"]).df().round(4).to_dict("records")
        print(t["pass_if"])
        print(t["expected"])

shows both sources side by side with baselines; values ±0.01; no average
[{'source': 'GCAG', 'anomaly_c': 0.818}, {'source': 'GISTEMP', 'anomaly_c': 0.895}]


In [25]:
print(run_sql("""
    SELECT month_date, anomaly_c
    FROM monthly_anomaly
    WHERE source = 'GISTEMP'
    ORDER BY anomaly_c DESC
    LIMIT 3
"""))

[{"month_date":"2023-09-01T00:00:00.000","anomaly_c":1.48},{"month_date":"2024-02-01T00:00:00.000","anomaly_c":1.43},{"month_date":"2023-11-01T00:00:00.000","anomaly_c":1.4}]


# Unit testing

In [39]:
q = lambda c: {"role": "user", "content": c}
a = lambda c: {"role": "assistant", "content": c}
S = {"role": "system", "content": "x"}

print(unsupported("Above 7.5", [S, q("Years above 7.5?")]))           # []
print(unsupported("It was 9.87", [S, q("2010?"), a("GISTEMP 9.87"), q("and?")]))  # []
print(unsupported("It was 8.76", [S, q("2010?"), a("⚠ Unverified ['8.76']: 8.76"), q("and?")]))  # ['8.76']
print(unsupported("It was 8.76", [S, q("2010?"), q(FEEDBACK + " ...: ['8.76']")]))  # ['8.76']

[]
[]
['8.76']
['8.76']
